# Prepare training set

In [ ]:
import os
import subprocess
import glob


In [ ]:
raw_data="../../data/species"
lyric_data="../../lyric_annotator"
isoquant_data="../../isoquant_annotator"

#annotation sources used as the geneid training base. A species may appear under
#more than one source to train/compare it from several origins.
reference=["Cyanidioschyzon_merolae_strain_10D_280699"]
lyric=["Babesia_duncani_323732", "Emiliania_huxleyi_CCMP1516_280463", "Cyanidiococcus_yangmingshanensis_2690220", "Hydrurus_foetidus_2996", "Chlorella_sorokiniana_3076"]
isoquant=["Chlamydomonas_reinhardtii_3055"]
baseAnn_origins={"reference":reference, "lyric":lyric, "isoquant":isoquant}

#ordered glob templates per source; {sp} is the full <Species>_<taxid> key from raw_data.
#first pattern that resolves to a single file wins (lyric: merged, then unmerged predictions).
#reference matches both RefSeq (GCF) and GenBank (GCA) annotation files.
src_glob={
    "reference": [raw_data+"/{sp}/GC*/{sp}_*_GC*.gff.gz"],
    "lyric":     [lyric_data+"/summary/merge/pred/{sp}_mergedRef.gff",
                  lyric_data+"/summary/pred/{sp}_pred.gff"],
    "isoquant":  [isoquant_data+"/summary/pred/{sp}_pred.gff"],
}

#composite work-unit key: <species>_<source>. Source names must contain no "_"
#so that key.rsplit("_",1) recovers (species, source).
def unit_key(sp, src):
    return f"{sp}_{src}"

def resolve_path(sp, src):
    """First src_glob pattern for (sp, src) that resolves to exactly one file, else None."""
    tried=[]
    for pat in src_glob[src]:
        pattern=pat.format(sp=sp)
        tried.append(pattern)
        matches=glob.glob(pattern)
        if len(matches)==1:
            return matches[0]
        if len(matches)>1:
            print(f"Ambiguous {src} annotation for {sp}:\n{matches}")
            return None
    print(f"No {src} annotation for {sp} (tried: {tried})")
    return None

def resolve_units():
    """(sp, src, ann_path) for every assigned species with a resolvable annotation."""
    units=[]
    for src, names in baseAnn_origins.items():
        for sp in names:
            path=resolve_path(sp, src)
            if path:
                units.append((sp, src, path))
    return units

!ls $raw_data

## Clean assemblies

In [ ]:
#genome is shared across a species' sources, so clean each assembly once
tr_sp=sorted({sp for sp, _, _ in resolve_units()})

for sp in tr_sp:

    ref_assembly=!realpath $raw_data/$sp/GC[AF]*/*_genomic*.fna*
    ref_assembly=ref_assembly[0]
    ref_assembly_name=ref_assembly.split("/")[-1].replace(".gz", "")
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/CLEAN_{ref_assembly_name}"
    print(result_file)
    
    !bash ../scripts/clean_ref.sh $ref_assembly > $result_file

## Extract CDS from annotations

In [ ]:
#one CDS extraction per (species, source) unit; source-tagged names coexist
for sp, src, ref_ann in resolve_units():
    print("##########################################################")
    print(f"{sp} -> {src}: {ref_ann}")

    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/CDS_{unit_key(sp, src)}.gff"
    print(result_file)

    !bash ../scripts/get_CDS.sh $ref_ann > $result_file

!ls -lh ../training_data/species/**/CDS*

## Sample n genes from CDS annotations

In [ ]:
n=1000
#sample each source's CDS file independently (one per unit)
for cds_path in sorted(glob.glob("../training_data/species/*/CDS_*.gff")):
    cds_name=os.path.basename(cds_path)
    sp=os.path.basename(os.path.dirname(cds_path))
    print(cds_name)
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/sample{n}_{cds_name}"
    print(result_file)

    !bash ../scripts/sample_CDS.sh $cds_path $n > $result_file

# Training commands

In [ ]:
n=1000

!mkdir -p ../job_commands
with open("../job_commands/total_training.txt", "w") as out:
    
    #create trained parameters folder command
    parameters_folder="../results/trainedParams"
    print(f"mkdir -p {parameters_folder}", file=out)
    #one training block per (species, source) unit, keyed off its sampled CDS file
    for sampleN_CDS in sorted(glob.glob(f"../training_data/species/*/sample{n}_CDS_*.gff")):
        sampleN_CDS=os.path.abspath(sampleN_CDS)
        #unit = <species>_<source>, recovered from the CDS filename
        unit=os.path.basename(sampleN_CDS).replace(f"sample{n}_CDS_", "").replace(".gff", "")
        sp=unit.rsplit("_", 1)[0]

        #shared cleaned genome for the species
        clean_ref=glob.glob(f"../training_data/species/{sp}/CLEAN_*.fna")
        if not clean_ref:
            print(f"No cleaned genome for {sp}")
            continue
        clean_ref=os.path.abspath(clean_ref[0])

        #set folder for results
        result_sp=f"../results/specie_logs/{unit}/"

        #training command using singularity container
        geneidtrainer_command=f"singularity run \
                            ~/software/geneidtrainerdocker.sif \
                            /scripts_geneid/geneidTRAINer4docker.pl \
                            -species {unit} \
                            -gff {sampleN_CDS} \
                            -fastas {clean_ref} \
                            -results {result_sp} \
                            -reduced no"
        geneidtrainer_command=geneidtrainer_command.replace("                             ", " ")

        #copy parameter files for clear access(+git)
        #move them to folder that will be in git, and link them to unit specific
        mvParam_command=f"mv {result_sp}*.optimized.param {parameters_folder}/"
        lnParam_command=f"ln -vf {parameters_folder}/{unit}.geneid.optimized.param {result_sp}"


        print(f'echo "Specie: {unit}"', file=out)
        print(geneidtrainer_command)
        print(geneidtrainer_command, file=out)
        print(mvParam_command, file=out)
        print(lnParam_command, file=out)

subprocess.run(["cp", "../scripts/trainer.sh", "../job_commands/"])
subprocess.run(["bash", "../scripts/missing_train.sh"])

#Execute generated commands to train geneId